# 02 – Entraînement du UNet

Entraînement sur l'image Sentinel-2 de **2021** avec les labels WorldCover 2021 (5 classes). Le modèle sera ensuite appliqué à 2017 et 2025 dans le notebook 03.

Je repars de l'architecture du tutoriel UNet du cours, avec plusieurs modifications :

- **normalisation** par percentiles 2–98 % calculés une seule fois sur l'image 2021 (et réutilisés pour 2017/2025), au lieu de diviser chaque patch par son max → sinon les valeurs ne sont pas comparables d'une année à l'autre
- **découpage train/val/test par blocs de 5 km** plutôt qu'aléatoire : deux patches voisins se ressemblent beaucoup, un découpage aléatoire surestimerait les scores
- **poids par classe** (en 1/√fréquence) car l'eau et les prairies sont peu représentées ; pixels sans donnée ignorés
- dropout 0,1 + augmentation (rotations / symétries), early stopping
- évaluation par **IoU par classe** et matrice de confusion plutôt que l'accuracy seule

## 1. Configuration

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow", tf.__version__, "| GPU :", tf.config.list_physical_devices("GPU"))

DATA_DIR = Path("data_saclay")
IMAGE_PATH = DATA_DIR / "s2_2021.tif"
LABEL_PATH = DATA_DIR / "labels_2021.tif"
OUT_DIR = Path("resultats")
OUT_DIR.mkdir(exist_ok=True)

NOMS_CLASSES = ["Bâti", "Forêt / arbres", "Cultures", "Prairies / autre", "Eau"]
N_CLASSES = len(NOMS_CLASSES)
IGNORE = 255

PATCH = 128          # taille des patches (pixels) : 128 px = 1,28 km
BLOC = 512           # taille des blocs géographiques pour le découpage (512 px = 5,12 km)
STRIDE_TRAIN = 64    # chevauchement des patches d'entraînement (plus d'exemples)
PARTS = (0.70, 0.15, 0.15)  # train / validation / test (en nombre de blocs)

BATCH_SIZE = 16
EPOCHS = 80          # maximum : l'early stopping arrêtera avant
LEARNING_RATE = 1e-3
DROPOUT = 0.1
BASE_FILTRES = 32    # 32-64-128-256-512 au lieu de 64-...-1024 dans le tuto (moins de données ici)
SEED = 42

tf.keras.utils.set_random_seed(SEED)

## 2. Chargement et normalisation

In [ ]:
with rasterio.open(IMAGE_PATH) as src:
    image = src.read().astype("float32")          # (4, H, W)
with rasterio.open(LABEL_PATH) as src:
    labels = src.read(1)                           # (H, W)

image = np.moveaxis(image, 0, -1)                  # (H, W, 4)
vide = (image == 0).all(axis=-1)                   # pixels sans donnée Sentinel-2
labels[vide] = IGNORE
H, W = labels.shape
print(f"Image : {H} x {W} pixels, {image.shape[-1]} bandes")

# Normalisation globale : percentiles 2 % et 98 % de chaque bande, sur les pixels valides
valides = image[~vide]
p_bas = np.percentile(valides, 2, axis=0)
p_haut = np.percentile(valides, 98, axis=0)
print("Percentiles 2 % :", p_bas.round(1))
print("Percentiles 98 % :", p_haut.round(1))

def normaliser(img):
    return np.clip((img - p_bas) / (p_haut - p_bas), 0, 1).astype("float32")

image_n = normaliser(image)
image_n[vide] = 0

# Sauvegarde : le notebook 3 devra appliquer EXACTEMENT la même normalisation à 2017 et 2025
with open(OUT_DIR / "normalisation.json", "w") as f:
    json.dump({"p_bas": p_bas.tolist(), "p_haut": p_haut.tolist()}, f, indent=2)

## 3. Découpage en blocs géographiques et extraction des patches

In [ ]:
n_by, n_bx = H // BLOC, W // BLOC
blocs = [(by, bx) for by in range(n_by) for bx in range(n_bx)]
rng = np.random.default_rng(SEED)
rng.shuffle(blocs)

n_train = int(PARTS[0] * len(blocs))
n_val = int(PARTS[1] * len(blocs))
blocs_split = {
    "train": blocs[:n_train],
    "val": blocs[n_train:n_train + n_val],
    "test": blocs[n_train + n_val:],
}
print({k: len(v) for k, v in blocs_split.items()}, "blocs de", BLOC * 10 / 1000, "km")


def extraire(liste_blocs, stride):
    X, Y = [], []
    for by, bx in liste_blocs:
        y0, x0 = by * BLOC, bx * BLOC
        for dy in range(0, BLOC - PATCH + 1, stride):
            for dx in range(0, BLOC - PATCH + 1, stride):
                lab = labels[y0 + dy:y0 + dy + PATCH, x0 + dx:x0 + dx + PATCH]
                if (lab == IGNORE).mean() > 0.5:
                    continue  # patch majoritairement vide
                X.append(image_n[y0 + dy:y0 + dy + PATCH, x0 + dx:x0 + dx + PATCH])
                Y.append(lab)
    return np.stack(X), np.stack(Y)


X_train, Y_train = extraire(blocs_split["train"], STRIDE_TRAIN)
X_val, Y_val = extraire(blocs_split["val"], PATCH)
X_test, Y_test = extraire(blocs_split["test"], PATCH)
print(f"Patches : train {len(X_train)} | validation {len(X_val)} | test {len(X_test)}")

In [ ]:
# Carte du découpage
carte = np.full((n_by, n_bx), -1)
for i, nom in enumerate(["train", "val", "test"]):
    for by, bx in blocs_split[nom]:
        carte[by, bx] = i

rgb = image_n[..., [2, 1, 0]][: n_by * BLOC, : n_bx * BLOC]
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb)
ax.imshow(np.kron(carte, np.ones((BLOC, BLOC))), cmap=ListedColormap(["#2c7bb6", "#fdae61", "#d7191c"]),
          alpha=0.35, vmin=0, vmax=2)
for nom, c in zip(["Entraînement", "Validation", "Test"], ["#2c7bb6", "#fdae61", "#d7191c"]):
    ax.plot([], [], "s", color=c, label=nom, markersize=12)
ax.legend(loc="lower right")
ax.set_title("Découpage géographique train / validation / test")
ax.axis("off")
plt.savefig(OUT_DIR / "decoupage_blocs.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Poids des classes

Poids ∝ 1/√fréquence : une pondération en 1/fréquence donnait un poids énorme à l'eau, très rare, la racine est un compromis.

In [ ]:
comptes = np.bincount(Y_train[Y_train != IGNORE].ravel(), minlength=N_CLASSES).astype("float64")
freq = comptes / comptes.sum()
poids = 1 / np.sqrt(np.maximum(freq, 1e-6))
poids = poids / poids.mean()

print(f"{'Classe':18s} {'Fréquence':>10s} {'Poids':>8s}")
for nom, f_, p_ in zip(NOMS_CLASSES, freq, poids):
    print(f"{nom:18s} {f_:10.1%} {p_:8.2f}")

POIDS_TF = tf.constant(poids, dtype=tf.float32)

## 5. Pipeline de données (augmentation + poids par pixel)

In [ ]:
def augmenter(x, y):
    y = y[..., tf.newaxis]
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x, y = tf.image.rot90(x, k), tf.image.rot90(y, k)
    flip = tf.random.uniform([]) > 0.5
    x = tf.cond(flip, lambda: tf.image.flip_left_right(x), lambda: x)
    y = tf.cond(flip, lambda: tf.image.flip_left_right(y), lambda: y)
    return x, y[..., 0]


def ajouter_poids(x, y):
    y = tf.cast(y, tf.int32)
    valide = tf.not_equal(y, IGNORE)
    y_sur = tf.where(valide, y, 0)                       # valeur quelconque, son poids sera 0
    w = tf.gather(POIDS_TF, y_sur) * tf.cast(valide, tf.float32)
    return x, y_sur, w


def dataset(X, Y, entrainement):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if entrainement:
        ds = ds.shuffle(len(X), seed=SEED).map(augmenter, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.map(ajouter_poids, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


ds_train = dataset(X_train, Y_train, entrainement=True)
ds_val = dataset(X_val, Y_val, entrainement=False)

## 6. Modèle UNet

In [ ]:
def bloc_conv(x, filtres):
    for _ in range(2):
        x = layers.Conv2D(filtres, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    return x


def unet(input_shape, n_classes, base=32, dropout=0.1):
    entree = layers.Input(input_shape)
    sauts, x = [], entree
    for niveau in range(4):                               # encodeur
        x = bloc_conv(x, base * 2 ** niveau)
        sauts.append(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(dropout)(x)
    x = bloc_conv(x, base * 16)                           # goulot
    for niveau in reversed(range(4)):                     # décodeur
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, sauts[niveau]])
        x = bloc_conv(x, base * 2 ** niveau)
    sortie = layers.Conv2D(n_classes, 1, activation="softmax")(x)
    return keras.Model(entree, sortie, name="unet_saclay")


model = unet((PATCH, PATCH, 4), N_CLASSES, base=BASE_FILTRES, dropout=DROPOUT)
model.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    weighted_metrics=["accuracy"],
)
print(f"Paramètres : {model.count_params():,}")

## 7. Entraînement

In [ ]:
import time

MODEL_PATH = OUT_DIR / "unet_saclay.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
]

t0 = time.time()
historique = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, callbacks=callbacks)
duree = time.time() - t0
n_ep = len(historique.history["loss"])
print(f"\nEntraînement terminé : {n_ep} époques en {duree / 60:.1f} min ({duree / n_ep:.0f} s par époque)")

In [ ]:
h = historique.history
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(h["loss"], label="Entraînement")
axes[0].plot(h["val_loss"], label="Validation")
axes[0].set_title("Loss (entropie croisée pondérée)")
axes[1].plot(h["accuracy"], label="Entraînement")
axes[1].plot(h["val_accuracy"], label="Validation")
axes[1].set_title("Accuracy pondérée")
for ax in axes:
    ax.set_xlabel("Époque")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "historique_entrainement.png", dpi=120)
plt.show()

## 8. Évaluation sur le test

Blocs de test jamais vus pendant l'entraînement. Métriques : IoU par classe (métrique standard en segmentation), F1, précision/rappel et matrice de confusion.

In [ ]:
proba = model.predict(X_test, batch_size=32, verbose=0)
pred = proba.argmax(-1)
valide = Y_test != IGNORE
vrai, predit = Y_test[valide].astype(int), pred[valide]

cm = np.bincount(N_CLASSES * vrai + predit, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
tp = np.diag(cm).astype(float)
precision = tp / np.maximum(cm.sum(0), 1)
rappel = tp / np.maximum(cm.sum(1), 1)
f1 = 2 * precision * rappel / np.maximum(precision + rappel, 1e-9)
iou = tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)

print(f"{'Classe':18s} {'IoU':>6s} {'F1':>6s} {'Précision':>10s} {'Rappel':>8s} {'Pixels test':>12s}")
for i, nom in enumerate(NOMS_CLASSES):
    print(f"{nom:18s} {iou[i]:6.3f} {f1[i]:6.3f} {precision[i]:10.3f} {rappel[i]:8.3f} {cm[i].sum():12,d}")
print(f"\nAccuracy globale : {tp.sum() / cm.sum():.3f}")
print(f"mIoU (moyenne des IoU) : {iou.mean():.3f}")

metriques = {
    "accuracy": float(tp.sum() / cm.sum()), "mIoU": float(iou.mean()),
    "par_classe": {nom: {"IoU": float(iou[i]), "F1": float(f1[i]),
                         "precision": float(precision[i]), "rappel": float(rappel[i])}
                   for i, nom in enumerate(NOMS_CLASSES)},
    "matrice_confusion": cm.tolist(),
}
with open(OUT_DIR / "metriques_test.json", "w", encoding="utf-8") as f:
    json.dump(metriques, f, indent=2, ensure_ascii=False)

In [ ]:
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(N_CLASSES), NOMS_CLASSES, rotation=35, ha="right")
ax.set_yticks(range(N_CLASSES), NOMS_CLASSES)
ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle (WorldCover)")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_title("Matrice de confusion (normalisée par ligne)")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(OUT_DIR / "matrice_confusion.png", dpi=120)
plt.show()

In [ ]:
# Quelques exemples visuels : image, vérité, prédiction
couleurs = ListedColormap(["#d7191c", "#1a9641", "#fdae61", "#a6d96a", "#2b83ba"])
idx = np.random.default_rng(0).choice(len(X_test), size=min(4, len(X_test)), replace=False)
fig, axes = plt.subplots(len(idx), 3, figsize=(10, 3.3 * len(idx)))
for ligne, i in zip(np.atleast_2d(axes), idx):
    ligne[0].imshow(np.clip(X_test[i][..., [2, 1, 0]] * 1.2, 0, 1))
    ligne[1].imshow(np.ma.masked_equal(Y_test[i], IGNORE), cmap=couleurs, vmin=0, vmax=4, interpolation="nearest")
    ligne[2].imshow(pred[i], cmap=couleurs, vmin=0, vmax=4, interpolation="nearest")
    for ax, titre in zip(ligne, ["Image 2021", "WorldCover", "Prédiction UNet"]):
        ax.set_title(titre)
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "exemples_test.png", dpi=120)
plt.show()

In [ ]:
config = {
    "image": str(IMAGE_PATH), "labels": str(LABEL_PATH), "classes": NOMS_CLASSES,
    "patch": PATCH, "bloc": BLOC, "stride_train": STRIDE_TRAIN, "parts": PARTS,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "dropout": DROPOUT,
    "base_filtres": BASE_FILTRES, "poids_classes": poids.tolist(),
    "epoques_effectuees": n_ep, "duree_min": round(duree / 60, 1),
    "n_patches": {"train": len(X_train), "val": len(X_val), "test": len(X_test)},
}
with open(OUT_DIR / "config_entrainement.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("Tout est enregistré dans", OUT_DIR)

## Observations

Premier entraînement court (10 époques, ~13 min sur M4 Max) : **mIoU = 0,72**, accuracy = 0,87.

| Classe | IoU |
|---|---|
| Bâti | 0,81 |
| Forêt | 0,80 |
| Cultures | 0,82 |
| Eau | 0,67 |
| Prairies / autre | 0,48 |

- Le bâti, la classe qui nous intéresse le plus, est bien reconnu (rappel 0,92). Les confusions bâti ↔ forêt viennent surtout des zones pavillonnaires : à 10 m un pixel mélange toits et jardins.
- Prairies / autre est la classe la plus faible : c'est une classe « fourre-tout » (pelouses, friches, sol nu…) confondue avec les cultures et la forêt.
- L'eau a un très bon rappel (0,97) mais une précision moyenne (0,68) : le modèle voit de l'eau sur certains toits sombres ou ombres en ville.
- Pendant les premières époques la validation était très mauvaise (val_accuracy ≈ 0,15) puis remonte brusquement vers l'époque 5 : probablement le temps que les statistiques de BatchNormalization se stabilisent.

À faire : entraînement complet (early stopping, max 80 époques).